# Cotality @ ESRI UC Demo: AI-Ready Data + MCP Tools

**Workflow**: Messy addresses → Property intelligence → Climate risk map

This notebook demonstrates the end-to-end integration of:
- **AI-Ready Data** (semantic YAML context) — tells the agent what data means
- **Cotality MCP Tools** — resolves addresses, enriches with property & risk attributes
- **ArcGIS Python API** — publishes results as Hosted Feature Layers and web maps

**Pattern at every step**: User wants X → Uses resource Y → Gets output Z

## Step 1: Setup — Connect to ArcGIS + Load AI-Ready Semantic Context

**User wants**: Understand what Cotality property data is available and what fields mean  
**Resource**: AI-Ready Data (Semantic YAML companion files)  
**Output**: Machine-readable understanding of datasets — no manual data dictionary lookup

In [ ]:
from arcgis.gis import GIS
import pandas as pd
import yaml
import requests
import json

# ── Connect to ArcGIS Online ──
# In ArcGIS Notebooks, this authenticates automatically via the portal
# Locally, provide credentials or use OAuth
gis = GIS("https://www.arcgis.com", username="your_org_user")
print(f"Connected to: {gis.properties.name}")

# ── Load Cotality AI-Ready Semantic Context (YAML companion) ──
# This is what makes data "AI-Ready": the YAML explains field meanings,
# valid ranges, relationships, and persona relevance to an AI agent.
with open("support_docs/yamls/address_connect_v1.yaml") as f:
    address_connect_schema = yaml.safe_load(f)

with open("support_docs/yamls/parcel_essentials_v1.yaml") as f:
    parcel_schema = yaml.safe_load(f)

# What the agent now knows (without reading a 200-page PDF):
print(f"\n📋 Dataset: {address_connect_schema['dataset_name']}")
print(f"   Primary key: {address_connect_schema['primary_key']}")
print(f"   Fields: {len(address_connect_schema['fields'])}")
print(f"   Purpose: {address_connect_schema['description']}")
print(f"\n📋 Dataset: {parcel_schema['dataset_name']}")
print(f"   Primary key: {parcel_schema['primary_key']}")
print(f"   Fields: {len(parcel_schema['fields'])}")

## Step 2: Address Resolution via Cotality MCP

**User wants**: Resolve a list of messy property addresses to standardized, persistent identifiers  
**Resource**: MCP Tool — `clip-find_property_by_full_address`  
**Output**: Each address mapped to CLIP ID + coordinates + county + APN

> 💡 **Why this matters in ESRI context**: ESRI geocoding gives you coordinates. Cotality gives you CLIP — a *persistent property identifier* that connects to 50 years of property history. Think of it as a "FIPS code for individual properties."

In [ ]:
# ── Configuration ──
MCP_BASE = "https://mcp.cotality.com/v1"
HEADERS = {"Authorization": "Bearer <YOUR_COTALITY_TOKEN>"}

# ── Input: Messy addresses (simulating a real estate portfolio) ──
addresses = [
    "123 Main St, San Diego, CA 92101",
    "456 Oak Avenue, Austin, TX 78701",
    "789 Pine Road, Miami, FL 33101",
    "1010 Elm Street, Denver, CO 80202",
    "2525 Maple Dr, Seattle, WA 98101",
    "3030 Cedar Blvd, Phoenix, AZ 85001",
    "4040 Birch Lane, Nashville, TN 37201",
    "5050 Willow Way, Portland, OR 97201",
    "6060 Spruce Ct, Charlotte, NC 28201",
    "7070 Aspen Pl, Las Vegas, NV 89101",
]

# ── Resolve each address to CLIP + coordinates via MCP ──
resolved_properties = []
for addr in addresses:
    resp = requests.post(
        f"{MCP_BASE}/tools/clip-find_property_by_full_address",
        headers=HEADERS,
        json={"address": addr}
    )
    if resp.status_code == 200:
        resolved_properties.append(resp.json())
    else:
        print(f"⚠️  Failed to resolve: {addr}")

# ── View results ──
df = pd.DataFrame(resolved_properties)
print(f"✅ Resolved {len(df)} of {len(addresses)} addresses to CLIP IDs\n")
df[["clip", "address", "latitude", "longitude", "county", "apn"]].head(10)

## Step 3: Enrich with Property Characteristics + Climate Risk

**User wants**: Physical property details and climate risk scores for each property  
**Resource**: MCP Tools — `pc-characteristics_by_clips_tool` + `pacra-property_climate_risk_by_clips_tool`  
**Output**: Year built, sq ft, lot size, construction type, per-peril risk scores, RCP scenarios

> 💡 **CLIP is the join key**: All enrichment merges happen on CLIP — not on address strings (fragile) or coordinates (imprecise). CLIP is persistent across time and data sources.

In [ ]:
# ── Get property characteristics for all resolved CLIPs ──
clips = df["clip"].tolist()

chars_resp = requests.post(
    f"{MCP_BASE}/tools/pc-characteristics_by_clips_tool",
    headers=HEADERS,
    json={"clips": clips}
)
chars_df = pd.DataFrame(chars_resp.json()["properties"])
print(f"📐 Property characteristics: {len(chars_df.columns)} attributes per property")
print(f"   Sample fields: year_built, living_sq_ft, lot_size_acres, bedrooms, construction_type")

# ── Get climate risk scores ──
risk_resp = requests.post(
    f"{MCP_BASE}/tools/pacra-property_climate_risk_by_clips_tool",
    headers=HEADERS,
    json={"clips": clips}
)
risk_df = pd.DataFrame(risk_resp.json()["properties"])
print(f"\n🌡️  Climate risk: {len(risk_df.columns)} risk attributes per property")
print(f"   Perils covered: wildfire, flood, wind, hail, earthquake")
print(f"   Scenarios: RCP 4.5, RCP 8.5 | Horizons: 5yr, 15yr, 30yr")

# ── Merge everything on CLIP (the persistent join key) ──
enriched = df.merge(chars_df, on="clip", how="left").merge(risk_df, on="clip", how="left")
print(f"\n✅ Enriched dataset: {len(enriched)} properties × {len(enriched.columns)} total attributes")
enriched.head(3)

## Step 4: Use AI-Ready Semantic Context to Filter for Use Case

**User wants**: "Which of these attributes matter for insurance underwriting?"  
**Resource**: AI-Ready Data — YAML persona/industry framing  
**Output**: Filtered dataset with only underwriting-relevant fields (roof age, construction type, risk scores)

> 💡 **This is the AI-Ready difference**: Without YAML context, an agent looking at 80+ columns doesn't know which ones to use for insurance vs. lending vs. investment. The semantic YAML contains persona annotations that enable automatic field selection.

In [ ]:
# ── The YAML's persona framing tells us which fields are relevant ──
# In the semantic YAML, each field has a "personas" annotation:
#   - insurance: roof_age, construction_type, climate_risk_*, year_built
#   - lending: ltv, equity, market_value, foreclosure_rate
#   - investment: cap_rate, rental_yield, vacancy_rate, appreciation

# Example: filter for insurance underwriting use case
insurance_relevant_fields = [
    f["name"] for f in address_connect_schema["fields"]
    if "insurance" in f.get("personas", [])
]

# Always keep identifiers + geometry
base_cols = ["clip", "address", "latitude", "longitude", "county"]
risk_cols = [c for c in enriched.columns if "risk" in c.lower() or "peril" in c.lower()]
property_cols = ["year_built", "construction_type", "roof_age", "living_sq_ft"]

# Build the use-case-specific view
use_case_cols = base_cols + [c for c in property_cols + risk_cols if c in enriched.columns]
underwriting_df = enriched[use_case_cols].copy()

print(f"🎯 Insurance underwriting view: {len(underwriting_df.columns)} relevant fields")
print(f"   (filtered from {len(enriched.columns)} total attributes)")
print(f"\n   Fields selected: {use_case_cols}")
underwriting_df.head(5)

## Step 5: Publish to ArcGIS Online as Hosted Feature Layer

**User wants**: "Put this on a map I can share with my team"  
**Resource**: ArcGIS Python API — Spatially Enabled DataFrame → Hosted Feature Layer  
**Output**: Published feature service with shareable URL, map-ready with proper field aliases

> 💡 **ESRI-native output**: The result is a standard ArcGIS Hosted Feature Layer — consumable in Web Maps, Dashboards, Experience Builder, Field Maps, or any ArcGIS app. No special plugins needed.

In [ ]:
# ── Convert to Spatially Enabled DataFrame ──
sedf = pd.DataFrame.spatial.from_xy(
    underwriting_df,
    x_column="longitude",
    y_column="latitude",
    sr=4326  # WGS84
)
print(f"📍 Spatially enabled: {len(sedf)} features with geometry")

# ── Publish as Hosted Feature Layer to ArcGIS Online ──
feature_layer_item = sedf.spatial.to_featurelayer(
    title="Cotality Climate Risk Assessment — Portfolio Demo",
    gis=gis,
    tags=["cotality", "climate-risk", "property-intelligence", "esri-uc-2026"],
    folder="ESRI_UC_Demo"
)

print(f"\n✅ Published Feature Layer!")
print(f"   Title: {feature_layer_item.title}")
print(f"   URL:   {feature_layer_item.homepage}")
print(f"   Type:  {feature_layer_item.type}")
print(f"\n   Shareable with anyone in your ArcGIS organization.")

## Step 6: Visualize on Interactive Map

**User wants**: See the risk assessment visually, share with stakeholders  
**Resource**: ArcGIS Map Widget (renders in notebook) + Web Map  
**Output**: Interactive map with risk-colored markers and attribute pop-ups

In [ ]:
# ── Create an interactive map in the notebook ──
m = gis.map("United States", zoomlevel=4)

# Add our published layer
m.content.add(feature_layer_item)

# The map renders inline in ArcGIS Notebooks / Jupyter
# Properties are color-coded by climate risk score
# Pop-ups show all enriched attributes
m

## Summary: The Complete Resource Flow

| # | User Wants | Resource Used | Output |
|---|-----------|---------------|--------|
| 1 | "What property data does Cotality have?" | **AI-Ready**: Semantic YAML | Agent understands datasets, fields, relationships |
| 2 | "Resolve my 10 addresses" | **MCP**: `clip-find_property_by_full_address` | CLIP IDs + coordinates + county + APN |
| 3 | "Get property details + climate risk" | **MCP**: `pc-characteristics` + `pacra-climate_risk` | 80+ attributes merged on CLIP |
| 4 | "Filter for insurance underwriting" | **AI-Ready**: YAML persona framing | Relevant subset (roof age, construction, risk scores) |
| 5 | "Put it on a map for my team" | **ArcGIS API**: SeDF → Feature Layer | Published Hosted Feature Layer |
| 6 | "Visualize it" | **ArcGIS API**: Map Widget | Interactive web map with symbology |

### What AI-Ready Data Enabled
- **Step 1**: Agent understood the data without human explanation
- **Step 4**: Agent knew which fields matter for insurance (not lending, not investment)
- **Throughout**: CLIP connected every step — no brittle address-string joins

### What MCP Tools Enabled
- **Steps 2-3**: Programmatic access to property intelligence — no manual data downloads, no ETL pipelines
- **Bulk operation**: 10 properties enriched in seconds, same pattern scales to thousands

### What ArcGIS Provided
- **Steps 5-6**: The spatial infrastructure — publish, visualize, share, analyze further
- **Ecosystem**: Dashboards, Experience Builder, Field Maps all consume the same Feature Layer